<a href="https://colab.research.google.com/github/kaguradance/AI/blob/main/train_and_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [8]:
!pip install pythainlp emoji beautifulsoup4 xgboost joblib

In [9]:
import re
import json
import pandas as pd
import joblib
import unicodedata
from bs4 import BeautifulSoup
import emoji

# NLP Libraries
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack

# XGBoost
from xgboost import XGBClassifier

In [10]:
thai_stops = thai_stopwords()

def clean_text(text):
    text = str(text)

    # Emoji → text
    text = emoji.demojize(text)
    text = text.replace(":", " ")

    text = unicodedata.normalize("NFC", text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"photo via [a-zA-Z0-9 ]+", "", text, flags=re.IGNORECASE)

    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9 ]", " ", text)

    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
    tokens = [w for w in tokens if w not in thai_stops and len(w) > 1]

    return " ".join(tokens)

In [11]:
from google.colab import files
uploaded = files.upload()

Saving train_sentiment.json to train_sentiment (2).json


In [12]:
with open("train_sentiment.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["text"] = df["text"].apply(clean_text)

X_text = df["text"]
y = df["sentiment"]

In [13]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [14]:
word_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=30000
)

Xw_train = word_vec.fit_transform(X_train_text)
Xc_train = char_vec.fit_transform(X_train_text)
X_train = hstack([Xw_train, Xc_train])

Xw_test = word_vec.transform(X_test_text)
Xc_test = char_vec.transform(X_test_text)
X_test = hstack([Xw_test, Xc_test])

In [20]:
svm = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", C=2.0)

xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1
)

stack_model = StackingClassifier(
    estimators=[
        ("svm", svm),
        ("lr", lr),
        ("xgb", xgb)
    ],
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

In [21]:
print("🚀 Training Model...")
stack_model.fit(X_train, y_train)

y_pred = stack_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

🚀 Training Model...
Accuracy: 0.7916666666666666
              precision    recall  f1-score   support

    negative       0.83      0.85      0.84       600
     neutral       0.73      0.70      0.71       600
    positive       0.81      0.82      0.82       600

    accuracy                           0.79      1800
   macro avg       0.79      0.79      0.79      1800
weighted avg       0.79      0.79      0.79      1800



In [22]:
joblib.dump({
    "model": stack_model,
    "word_vec": word_vec,
    "char_vec": char_vec
}, "sentiment_bundle3.pkl")

print("✅ Training completed and all components saved")

✅ Training completed and all components saved


In [23]:
files.download("sentiment_bundle3.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>